
## CONFIGURACIÓN Y CARGA DE DATOS

In [1]:
!pip install pandas numpy matplotlib seaborn scikit-learn plotly pymongo
!pip install jupyter
!pip install ipywidgets  # Para widgets interactivos
!pip install nbformat  # Para formateo de notebooks

  Using cached joblib-1.5.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/10.7 MB 13.0 MB/s eta 0:00:01
   ----------- ---------------------------- 3.1/10.7 MB 10.8 MB/s eta 0:00:01
   ------------------ --------------------- 5.0/10.7 MB 10.4 MB/s eta 0:00:01
   --------------------------- ------------ 7.3/10.7 MB 10.4 MB/s eta 0:00:01
   ------------------------------------- -- 10.0/10.7 MB 11.0 MB/s eta 0:00:01
   ---------------------------------------- 10.7/10.7 MB 10.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   ----- ---------------------------------- 2.1/16.3 MB 10.6 MB/s eta 0:00:02
   ---------- ----------------------------- 4.2/16.3 MB 10.8 MB/s eta 0:00:02
   ---------------- ----------------------- 6.6/16.3 MB 10.9 MB/s eta 0:00:01
   ---------------------


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 15.5 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pymongo import MongoClient
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, confusion_matrix
import plotly.express as px
import plotly.graph_objects as go

# Conexión a MongoDB
client = MongoClient('mongodb://143.198.59.209:27017/')
db = client['traffic_analysis']

## CARGA Y DESCRIPCIÓN DEL DATASET

In [ ]:
def load_traffic_data():
    """Carga los datos de MongoDB y los convierte en DataFrame"""
    videos = list(db.videos.find())

    # Crear DataFrame con las detecciones
    data = []
    for video in videos:
        for frame in video['frames']:
            for detection in frame['detections']:
                data.append({
                    'video_id': video['filename'],
                    'camera_id': video['camera_id'],
                    'frame_number': frame['frame_number'],
                    'timestamp': frame['timestamp'],
                    'confidence': detection['confidence'],
                    'bbox_x1': detection['bbox'][0],
                    'bbox_y1': detection['bbox'][1],
                    'bbox_x2': detection['bbox'][2],
                    'bbox_y2': detection['bbox'][3],
                    'vehicle_size': (detection['bbox'][2] - detection['bbox'][0]) *
                                  (detection['bbox'][3] - detection['bbox'][1])
                })

    return pd.DataFrame(data)
# Cargar datos
df = load_traffic_data()
print("Dimensiones del dataset:", df.shape)
print("\nPrimeras filas:")
display(df.head())


## ANÁLISIS EXPLORATORIO

In [ ]:
print("\nEstadísticas descriptivas:")
display(df.describe())

Visualizaciones

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='confidence', bins=30)
plt.title('Distribución de Confianza en las Detecciones')
plt.show()

Tamaño de vehículos por cámara

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='camera_id', y='vehicle_size')
plt.title('Distribución del Tamaño de Vehículos por Cámara')
plt.show()

## PREPARACIÓN DE DATOS

In [ ]:
print("\nValores nulos por columna:")
display(df.isnull().sum())

Codificación de variables categóricas

In [ ]:
le = LabelEncoder()
df['camera_id_encoded'] = le.fit_transform(df['camera_id'])

Normalización

In [ ]:
scaler = StandardScaler()
numeric_cols = ['confidence', 'vehicle_size', 'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2']
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

## MODELOS DE MACHINE LEARNING

In [ ]:
X = df[['confidence', 'vehicle_size', 'camera_id_encoded']]
y = df['camera_id']  # Clasificar por cámara

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


Random Forest

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
print("\nMétricas Random Forest:")
print(classification_report(y_test, rf_pred))

K-Means Clustering

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)
df['cluster'] = kmeans.fit_predict(X)
# Visualización de clusters
fig = px.scatter_3d(df, x='confidence', y='vehicle_size', z='camera_id_encoded',
                    color='cluster', title='Clusters de Vehículos')
fig.show()


## Analasis Temporal

Patrones por hora

In [ ]:
df['hour'] = pd.to_datetime(df['timestamp']).dt.hour
hourly_counts = df.groupby('hour').size()

plt.figure(figsize=(12, 6))
hourly_counts.plot(kind='bar')
plt.title('Detecciones por Hora del Día')
plt.xlabel('Hora')
plt.ylabel('Número de Detecciones')
plt.show()

## CONCLUSIONES

In [ ]:
"""Conclusiones técnicas:

1. Patrones de Tráfico:
   - Se observan picos de tráfico en las horas [X, Y, Z]
   - La cámara [X] muestra mayor densidad de vehículos

2. Precisión de Detección:
   - Confianza promedio: [X]%
   - Mejor rendimiento en condiciones [X]

3. Clasificación:
   - Random Forest alcanza [X]% de precisión
   - Mejor clasificación para cámara [X]

4. Clusters:
   - Se identificaron [X] patrones principales de vehículos
   - Cluster [X] representa vehículos [características]

5. Recomendaciones:
   - Ajustar sensibilidad en cámara [X]
   - Considerar recalibración en horas [X]
"""